# Step 2: Build Company Question Index

**What this notebook does:**
1. Walks all 658 company folders in `data/leetcode-companywise-interview-questions/`
2. For each company, reads up to 5 time-window CSV files
3. Merges (company, problem) pairs — tracking which time windows each question appears in
4. Extracts slug from the URL column
5. Writes `data/my/company_questions.jsonl` — one record per (company, problem) pair

**CSV format confirmed:** `ID, URL, Title, Difficulty, Acceptance %, Frequency %`

**Time windows available per company:**
- `thirty-days.csv` → `30d`
- `three-months.csv` → `3m`
- `six-months.csv` → `6m`
- `more-than-six-months.csv` → `6m+`
- `all.csv` → `all` (master list — frequency used as the canonical frequency)

In [1]:
# Cell 1 — Imports & Paths
import csv
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:/PRAKSHIT/VS CODE/Prep Assistant Project/placed_in")
COMPANY_DIR  = PROJECT_ROOT / "data" / "leetcode-companywise-interview-questions"
OUTPUT_FILE  = PROJECT_ROOT / "data" / "my" / "company_questions.jsonl"

# Time window filenames → tag mapping
WINDOWS = {
    "thirty-days.csv"          : "30d",
    "three-months.csv"         : "3m",
    "six-months.csv"           : "6m",
    "more-than-six-months.csv" : "6m+",
    "all.csv"                  : "all",
}

# Collect all valid company directories (skip hidden/non-dir entries)
company_dirs = sorted(
    d for d in COMPANY_DIR.iterdir()
    if d.is_dir() and not d.name.startswith(".")
)

print(f"Company dir   : {COMPANY_DIR}")
print(f"Exists        : {COMPANY_DIR.exists()}")
print(f"Total companies: {len(company_dirs)}")
print(f"Sample companies: {[d.name for d in company_dirs[:8]]}")

Company dir   : C:\PRAKSHIT\VS CODE\Prep Assistant Project\placed_in\data\leetcode-companywise-interview-questions
Exists        : True
Total companies: 657
Sample companies: ['1kosmos', '6sense', 'accelya', 'accenture', 'accolite', 'acko', 'acorns', 'activision']


In [3]:
# Cell 2 — Parse All Company CSVs
# Key: (company_name, problem_id) → record dict
all_records: dict = {}
parse_errors: list = []
files_processed = 0
files_skipped   = 0

for company_dir in company_dirs:
    company = company_dir.name

    for csv_filename, window_tag in WINDOWS.items():
        csv_path = company_dir / csv_filename
        if not csv_path.exists():
            files_skipped += 1
            continue

        files_processed += 1
        try:
            with csv_path.open("r", encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    try:
                        pid = int(row["ID"])
                        url = row.get("URL", "").rstrip("/")
                        slug = url.split("/")[-1] if url else ""
                        acc_str  = row.get("Acceptance %", "0%").strip().rstrip("%")
                        freq_str = row.get("Frequency %",  "0%").strip().rstrip("%")
                        acceptance = float(acc_str)  if acc_str  else 0.0
                        frequency  = float(freq_str) if freq_str else 0.0

                        key = (company, pid)

                        if key not in all_records:
                            all_records[key] = {
                                "company"    : company,
                                "problemId"  : pid,
                                "slug"       : slug,
                                "title"      : row.get("Title", "").strip(),
                                "difficulty" : row.get("Difficulty", "").strip(),
                                "acceptance" : acceptance,
                                "frequency"  : 0.0,   # will be set from all.csv
                                "windows"    : [],
                            }

                        rec = all_records[key]

                        # Use frequency from all.csv as the canonical frequency
                        if window_tag == "all":
                            rec["frequency"] = frequency
                            # Also fill acceptance from all.csv if not already set
                            rec["acceptance"] = acceptance

                        # Avoid duplicate window tags (some CSVs may overlap)
                        if window_tag not in rec["windows"]:
                            rec["windows"].append(window_tag)

                    except (ValueError, KeyError) as e:
                        parse_errors.append(f"{company}/{csv_filename}: {e}")

        except Exception as e:
            parse_errors.append(f"{company}/{csv_filename}: FILE ERROR: {e}")

print(f"Files processed : {files_processed:,}")
print(f"Files skipped   : {files_skipped:,} (company doesn't have that window)")
print(f"Total (company, problem) pairs: {len(all_records):,}")
print(f"Parse errors    : {len(parse_errors)}")
if parse_errors:
    print(f"  First 5: {parse_errors[:5]}")

Files processed : 1,648
Files skipped   : 1,637 (company doesn't have that window)
Total (company, problem) pairs: 17,666
Parse errors    : 0


In [4]:
# Cell 3 — Quick Stats Before Writing
df_preview = pd.DataFrame(list(all_records.values()))

print("Top 10 companies by problem count:")
top_companies = df_preview.groupby("company").size().sort_values(ascending=False).head(10)
print(top_companies.to_string())

print(f"\nProblems per company:")
size_stats = df_preview.groupby("company").size()
print(f"  Min     : {size_stats.min()}")
print(f"  Max     : {size_stats.max()}")
print(f"  Average : {size_stats.mean():.0f}")
print(f"  Median  : {size_stats.median():.0f}")

print(f"\nDifficulty split:")
print(df_preview["difficulty"].value_counts().to_string())

print(f"\nWindow coverage:")
for win in ["30d", "3m", "6m", "6m+", "all"]:
    count = df_preview["windows"].apply(lambda ws: win in ws).sum()
    print(f"  {win:<6}: {count:,} (company, problem) pairs")

Top 10 companies by problem count:
company
google           2274
amazon           1957
microsoft        1374
meta             1366
bloomberg        1190
uber              366
tiktok            359
oracle            323
apple             303
goldman-sachs     259

Problems per company:
  Min     : 1
  Max     : 2274
  Average : 27
  Median  : 4

Difficulty split:
difficulty
Medium    10096
Easy       4335
Hard       3235

Window coverage:
  30d   : 399 (company, problem) pairs
  3m    : 1,771 (company, problem) pairs
  6m    : 3,851 (company, problem) pairs
  6m+   : 15,691 (company, problem) pairs
  all   : 17,641 (company, problem) pairs


In [5]:
# Cell 4 — Write Output JSONL
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for rec in all_records.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

file_mb = OUTPUT_FILE.stat().st_size / 1024 / 1024
print(f"✅ Wrote {len(all_records):,} records to:")
print(f"   {OUTPUT_FILE}")
print(f"   File size : {file_mb:.1f} MB")

# Sample output records
print("\nSample records (Amazon, first 3):")
amazon_recs = [r for r in all_records.values() if r["company"] == "amazon"][:3]
for r in amazon_recs:
    print(f"  #{r['problemId']:>4} {r['slug']:<50} freq={r['frequency']:.1f}% windows={r['windows']}")

✅ Wrote 17,666 records to:
   C:\PRAKSHIT\VS CODE\Prep Assistant Project\placed_in\data\my\company_questions.jsonl
   File size : 3.5 MB

Sample records (Amazon, first 3):
  #   1 two-sum                                            freq=100.0% windows=['30d', '3m', '6m', '6m+', 'all']
  #   3 longest-substring-without-repeating-characters     freq=87.5% windows=['30d', '3m', '6m', '6m+', 'all']
  # 121 best-time-to-buy-and-sell-stock                    freq=87.5% windows=['30d', '3m', '6m', '6m+', 'all']
